# Topic: SQL: GROUP BY + SUM/AVG

## Definition (30-second explanation)
* **GROUP BY** splits data into groups based on unique values in a categorical column.
* **SUM** calculates the total sum of a numeric column within each group.
* **AVG** computes the mean value by dividing the sum by the number of non-NULL rows in the group.
* **HAVING** is used to filter the resulting dataset based on these aggregated values, operating *after* the aggregation occurs.

## Why Interviewers Ask This
* It is one of the most frequent patterns in data analyst and data science interviews.
* It tests your understanding of the SQL logical order of operations (specifically filtering before vs. after aggregation).
* It verifies you understand how SQL handles `NULL` values by default during math operations.

## Core Concepts
* `SUM()` and `AVG()` both automatically ignore `NULL` values by default.
* `HAVING` filters on the result of aggregate functions, whereas `WHERE` filters individual rows before aggregation.
* You can combine `COUNT()`, `SUM()`, and `AVG()` simultaneously in a single query.

## When to Use
* When finding total metrics per group (e.g., total revenue per region, total ad spend per campaign).
* When calculating average values per group (e.g., average salary per department, average session duration per device).
* When you need to filter aggregated results (using `HAVING`).

## Advantages
* Condenses massive transactional tables into readable summary tables for business analysis.
* Allows simultaneous calculation of multiple metrics (sum, mean, count) over the same grouped dimensions.

## Limitations
* `SUM` and `AVG` can only be applied to numeric columns; using them on non-numeric columns causes an error.
* Implicitly ignoring `NULL` values can sometimes surprise you if you expected them to count as zeros.

## Common Comparisons
* **WHERE vs. HAVING:** `WHERE` filters rows *before* aggregation. `HAVING` filters groups *after* aggregation.
* **SUM(column) vs. COUNT(*):** `SUM` adds up the actual numeric values inside a column, while `COUNT` just counts the number of rows.

## Common Interview Traps
* Using `WHERE` instead of `HAVING` to filter aggregated results (e.g., `WHERE AVG(salary) > 70000`).
* Forgetting to include all non-aggregated columns from the `SELECT` clause in the `GROUP BY` clause.
* Confusing `SUM()` with `COUNT()` when a question asks for "total value" rather than "total rows".

## SQL Syntax
```sql
SELECT 
    department, 
    SUM(salary) AS total_salary,
    ROUND(AVG(salary), 2) AS avg_salary -- rounds to 2 decimal places
FROM employees
GROUP BY department
HAVING AVG(salary) > 70000
ORDER BY avg_salary DESC;
```


## 45-Second Interview Answer

"The GROUP BY clause combined with SUM and AVG is essential for multidimensional analysis, like finding total and average revenue per region. A key concept to remember is the order of execution: WHERE filters raw data before grouping, while HAVING filters the aggregated results after grouping. Additionally, both SUM and AVG naturally ignore NULL values, which is an important edge case in data cleaning."

## Example Questions and Answers

### Q1. Find the total quantity sold per product.
* **Ideal Answer:** 
  ```sql
  SELECT product, SUM(quantity) AS total_quantity 
  FROM sales 
  GROUP BY product;
  ```
* **Common Mistake:** Using `COUNT(quantity)` instead of `SUM(quantity)`.
* **Follow-up:** How would you only show products that sold more than 10 units in total? *(Answer: Add `HAVING SUM(quantity) > 10`)*.

### Q2. Calculate the average order value per customer.
* **Ideal Answer:**
  ```sql
  SELECT customer_id, AVG(revenue) AS avg_order_value 
  FROM sales 
  GROUP BY customer_id;
  ```
* **Common Mistake:** Forgetting to group by `customer_id` resulting in a syntax error.
* **Follow-up:** What function would you use to ensure the average is returned as a cleaner output, like currency? *(Answer: Wrap the `AVG` function in a `ROUND(AVG(revenue), 2)`)*.

### Q3. Find the department with the highest total salary budget.
* **Ideal Answer:**
  ```sql
  SELECT department, SUM(salary) AS total_budget 
  FROM employees 
  GROUP BY department 
  ORDER BY total_budget DESC 
  LIMIT 1;
  ```
* **Common Mistake:** Trying to use `MAX(SUM(salary))` which isn't valid standard SQL without subqueries or specific dialect functions.
* **Follow-up:** What if two departments tied for the highest budget? *(Answer: I would use a window function like `RANK()` or `DENSE_RANK()` over the summed salaries)*.

### Q4. Show all regions where the total revenue exceeds 5000.
* **Ideal Answer:**
  ```sql
  SELECT region, SUM(revenue) AS total_revenue 
  FROM sales 
  GROUP BY region 
  HAVING SUM(revenue) > 5000;
  ```
* **Common Mistake:** Putting `SUM(revenue) > 5000` in the `WHERE` clause.
* **Follow-up:** Can you alias `SUM(revenue)` as `total` and use `HAVING total > 5000`? *(Answer: In some engines like MySQL/Postgres it works, but in standard SQL and Oracle/SQL Server, you cannot use aliases in the HAVING clause because of the order of execution)*.

### Q5. Calculate the total and average revenue per month for the year 2025.
* **Ideal Answer:**
  ```sql
  SELECT 
      EXTRACT(MONTH FROM sale_date) AS month, 
      SUM(revenue) AS total_revenue, 
      AVG(revenue) AS avg_revenue 
  FROM sales 
  WHERE EXTRACT(YEAR FROM sale_date) = 2025 
  GROUP BY EXTRACT(MONTH FROM sale_date);
  ```
* **Common Mistake:** Applying the year filter (2025) in the `HAVING` clause instead of `WHERE`.
* **Follow-up:** Why is it more performant to filter for the year 2025 using `WHERE` rather than `HAVING`? *(Answer: `WHERE` reduces the dataset size before the costly grouping and aggregation steps occur)*.

## Practice Questions:

In [1]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('/home/shail/interview-prep/01_SQL/oracle_hr.db')

### Q1: 
**Mock Schema (`employees` table):**
* `emp_id` (INT)
* `department` (VARCHAR)
* `salary` (DECIMAL)
* `hire_date` (DATE)

**Question:**
Write a SQL query to find the departments that have an average salary greater than $80,000, but *only* consider salaries of employees hired *after* '2020-01-01'. Please return the department name and the rounded average salary (to 2 decimal places).

Answer: (As HR schema is not used here, query can not be run. Same kind of quesry on HR schema with output below):
```sql
SELECT 
      department, 
      ROUND(AVG(salary), 2) AS avg_salary
  FROM employees
  WHERE hire_date > '2020-01-01'
  GROUP BY department
  HAVING AVG(salary) > 80000;;
```

In [16]:
pd.read_sql_query(sql= """
select department_id, round(avg(salary), 2) as avg_salary
from employees
where hire_date > '2016-01-01'
group by department_id
having round(avg(salary), 2) > 5000;
""", con= conn)

,department_id,avg_salary
0,NaN,7000.00
1,60.0,6000.00
2,80.0,7931.58
3,100.0,7350.00
